In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

BASE_DIR = Path("..")
PROC_DIR = BASE_DIR / "data" / "processed"
FIG_DIR  = BASE_DIR / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

master = pd.read_csv(PROC_DIR / "training_table.csv")

all_cols    = master.columns.tolist()
morgan_cols = [c for c in all_cols if c.startswith('morgan_')]
morpho_cols = [c for c in all_cols if c.startswith('Cells_') 
               or c.startswith('Nuclei_') 
               or c.startswith('Cytoplasm_')]
ic50_cols   = [c for c in all_cols 
               if c not in morgan_cols + morpho_cols + ['drug_name', 'Metadata_JCP2022']]

print(f"Drugs:               {len(master)}")
print(f"Morgan features:     {len(morgan_cols)}")
print(f"Morphology features: {len(morpho_cols)}")
print(f"IC50 columns:        {len(ic50_cols)}")
print(f"\nSample IC50 cols: {ic50_cols[:3]}")
print(f"Sample morpho cols: {morpho_cols[:3]}")

Drugs:               175
Morgan features:     2048
Morphology features: 3178
IC50 columns:        969

Sample IC50 cols: ['22RV1', '23132-87', '42-MG-BA']
Sample morpho cols: ['Cells_AreaShape_Area', 'Cells_AreaShape_BoundingBoxArea', 'Cells_AreaShape_BoundingBoxMaximum_X']


In [4]:
# build the drug-level dataset in long format
# Convert wide master table to long format: one row per (drug, cell_line) pair
ic50_long = master[['drug_name'] + ic50_cols].melt(
    id_vars='drug_name',
    var_name='cell_line',
    value_name='ln_ic50'
)

# Drop missing IC50 values
ic50_long = ic50_long.dropna(subset=['ln_ic50']).reset_index(drop=True)

print(f"Total (drug, cell_line) pairs: {len(ic50_long):,}")
print(f"Missing IC50 dropped:          {175*969 - len(ic50_long):,}")
print(f"\nSample:")
print(ic50_long.head(3).to_string())

Total (drug, cell_line) pairs: 149,679
Missing IC50 dropped:          19,896

Sample:
    drug_name cell_line   ln_ic50
0   Gefitinib     22RV1  4.315802
1  Vorinostat     22RV1  0.913749
2   Nilotinib     22RV1  4.238712


In [6]:
# train, val, test split by drug

# split by drug — not by cell line — to avoid data leakage
# test set drugs never seen during training
drugs = master['drug_name'].values
n_drugs = len(drugs)

np.random.seed(42)
shuffled_idx = np.random.permutation(n_drugs)

n_test  = int(n_drugs * 0.15)
n_val   = int(n_drugs * 0.15)
n_train = n_drugs - n_test - n_val

train_drugs = set(drugs[shuffled_idx[:n_train]])
val_drugs   = set(drugs[shuffled_idx[n_train:n_train+n_val]])
test_drugs  = set(drugs[shuffled_idx[n_train+n_val:]])

train_df = ic50_long[ic50_long['drug_name'].isin(train_drugs)].reset_index(drop=True)
val_df   = ic50_long[ic50_long['drug_name'].isin(val_drugs)].reset_index(drop=True)
test_df  = ic50_long[ic50_long['drug_name'].isin(test_drugs)].reset_index(drop=True)

print(f"Train drugs: {len(train_drugs)} → {len(train_df):,} pairs")
print(f"Val drugs:   {len(val_drugs)}  → {len(val_df):,} pairs")
print(f"Test drugs:  {len(test_drugs)}  → {len(test_df):,} pairs")
print(f"\nNo overlap: {len(train_drugs & test_drugs) == 0}")

Train drugs: 123 → 103,319 pairs
Val drugs:   26  → 22,677 pairs
Test drugs:  26  → 23,683 pairs

No overlap: True


In [7]:
# build feature matrices & normalize

# build feature lookup: drug_name → feature vector
morgan_lookup = master.set_index('drug_name')[morgan_cols].values
morpho_lookup = master.set_index('drug_name')[morpho_cols].values
drug_index    = {name: i for i, name in enumerate(master['drug_name'].values)}

# Fit scalers on training drugs only
train_drug_idx = [drug_index[d] for d in train_drugs]

morpho_scaler = StandardScaler()
morpho_scaler.fit(morpho_lookup[train_drug_idx])

morgan_scaled = morgan_lookup  # binary — no scaling needed
morpho_scaled = morpho_scaler.transform(morpho_lookup)

print(f"Morgan matrix:    {morgan_scaled.shape}")
print(f"Morphology matrix:{morpho_scaled.shape}")
print(f"\nMorphology mean (should be ~0 for train): "
      f"{morpho_scaled[train_drug_idx].mean():.4f}")
print(f"Morphology std  (should be ~1 for train): "
      f"{morpho_scaled[train_drug_idx].std():.4f}")

Morgan matrix:    (175, 2048)
Morphology matrix:(175, 3178)

Morphology mean (should be ~0 for train): -0.0000
Morphology std  (should be ~1 for train): 1.0000


In [8]:
# PyTorch Dataset & DataLoader

class DrugResponseDataset(Dataset):
    def __init__(self, df, drug_index, morgan_scaled, morpho_scaled):
        self.drug_idx = torch.tensor(
            [drug_index[d] for d in df['drug_name']], dtype=torch.long
        )
        self.targets = torch.tensor(df['ln_ic50'].values, dtype=torch.float32)
        self.morgan  = torch.tensor(morgan_scaled, dtype=torch.float32)
        self.morpho  = torch.tensor(morpho_scaled, dtype=torch.float32)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        idx = self.drug_idx[i]
        return self.morgan[idx], self.morpho[idx], self.targets[i]

train_ds = DrugResponseDataset(train_df, drug_index, morgan_scaled, morpho_scaled)
val_ds   = DrugResponseDataset(val_df,   drug_index, morgan_scaled, morpho_scaled)
test_ds  = DrugResponseDataset(test_df,  drug_index, morgan_scaled, morpho_scaled)

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False)

# Sanity check
morgan_batch, morpho_batch, target_batch = next(iter(train_loader))
print(f"Morgan batch:  {morgan_batch.shape}")
print(f"Morpho batch:  {morpho_batch.shape}")
print(f"Target batch:  {target_batch.shape}")
print(f"\nTarget range: [{target_batch.min():.2f}, {target_batch.max():.2f}]")

Morgan batch:  torch.Size([512, 2048])
Morpho batch:  torch.Size([512, 3178])
Target batch:  torch.Size([512])

Target range: [-5.99, 9.18]


In [9]:
# define the baseline MLP

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 256, 128], dropout=0.3):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(in_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_model(model, train_loader, val_loader, epochs=50, lr=1e-3, patience=10):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5, verbose=False
    )
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_state    = None
    patience_counter = 0
    history = {'train': [], 'val': []}

    for epoch in range(epochs):
        # Train
        model.train()
        train_loss = 0
        for morgan_b, morpho_b, target_b in train_loader:
            optimizer.zero_grad()
            out  = model(morgan_b)
            loss = criterion(out, target_b)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(target_b)
        train_loss /= len(train_loader.dataset)

        # Validate
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for morgan_b, morpho_b, target_b in val_loader:
                out  = model(morgan_b)
                loss = criterion(out, target_b)
                val_loss += loss.item() * len(target_b)
        val_loss /= len(val_loader.dataset)

        history['train'].append(train_loss)
        history['val'].append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

        if (epoch+1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d} | train MSE: {train_loss:.4f} | val MSE: {val_loss:.4f}")

    model.load_state_dict(best_state)
    return model, history


def evaluate_model(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for morgan_b, morpho_b, target_b in loader:
            out = model(morgan_b)
            preds.extend(out.numpy())
            targets.extend(target_b.numpy())
    preds   = np.array(preds)
    targets = np.array(targets)
    r, _    = pearsonr(preds, targets)
    rmse    = np.sqrt(((preds - targets)**2).mean())
    return r, rmse, preds, targets


print("MLP defined.")
print(f"Structure-only input dim: {len(morgan_cols)}")
print(f"Morphology-only input dim: {len(morpho_cols)}")
print(f"Concatenation input dim: {len(morgan_cols) + len(morpho_cols)}")

MLP defined.
Structure-only input dim: 2048
Morphology-only input dim: 3178
Concatenation input dim: 5226


In [10]:
# train structure-only baseline

print("="*50)
print("Condition 1: Structure-only (Morgan fingerprints)")
print("="*50)

model_morgan = MLP(input_dim=len(morgan_cols))

# Override forward to use only morgan features
class MorganMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = MLP(input_dim=len(morgan_cols))
    def forward(self, morgan, morpho):
        return self.mlp(morgan)

class MorphoMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = MLP(input_dim=len(morpho_cols))
    def forward(self, morgan, morpho):
        return self.mlp(morpho)

class ConcatMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = MLP(input_dim=len(morgan_cols) + len(morpho_cols))
    def forward(self, morgan, morpho):
        x = torch.cat([morgan, morpho], dim=1)
        return self.mlp(x)

# Redefine train/eval to accept both inputs
def train_model_v2(model, train_loader, val_loader, epochs=50, lr=1e-3, patience=10):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = nn.MSELoss()
    best_val_loss = float('inf')
    best_state    = None
    patience_counter = 0
    history = {'train': [], 'val': []}

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for morgan_b, morpho_b, target_b in train_loader:
            optimizer.zero_grad()
            out  = model(morgan_b, morpho_b)
            loss = criterion(out, target_b)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(target_b)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for morgan_b, morpho_b, target_b in val_loader:
                out      = model(morgan_b, morpho_b)
                val_loss += criterion(out, target_b).item() * len(target_b)
        val_loss /= len(val_loader.dataset)

        history['train'].append(train_loss)
        history['val'].append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_state       = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

        if (epoch+1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d} | train MSE: {train_loss:.4f} | val MSE: {val_loss:.4f}")

    model.load_state_dict(best_state)
    return model, history

def evaluate_model_v2(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for morgan_b, morpho_b, target_b in loader:
            out = model(morgan_b, morpho_b)
            preds.extend(out.numpy())
            targets.extend(target_b.numpy())
    preds   = np.array(preds)
    targets = np.array(targets)
    r, _    = pearsonr(preds, targets)
    rmse    = np.sqrt(((preds - targets)**2).mean())
    return r, rmse, preds, targets

# Train structure-only
print("\nTraining...")
model_morgan = MorganMLP()
model_morgan, hist_morgan = train_model_v2(model_morgan, train_loader, val_loader, epochs=100)
r_morgan, rmse_morgan, _, _ = evaluate_model_v2(model_morgan, test_loader)
print(f"\nStructure-only  | Pearson R: {r_morgan:.4f} | RMSE: {rmse_morgan:.4f}")

Condition 1: Structure-only (Morgan fingerprints)

Training...
  Epoch  10 | train MSE: 2.3571 | val MSE: 3.8838
  Early stopping at epoch 11

Structure-only  | Pearson R: 0.3333 | RMSE: 2.4652


In [11]:
print("="*50)
print("Condition 2: Morphology-only")
print("="*50)

print("\nTraining...")
model_morpho = MorphoMLP()
model_morpho, hist_morpho = train_model_v2(model_morpho, train_loader, val_loader, epochs=100)
r_morpho, rmse_morpho, _, _ = evaluate_model_v2(model_morpho, test_loader)
print(f"\nMorphology-only | Pearson R: {r_morpho:.4f} | RMSE: {rmse_morpho:.4f}")

Condition 2: Morphology-only

Training...
  Epoch  10 | train MSE: 2.3760 | val MSE: 4.0188
  Early stopping at epoch 14

Morphology-only | Pearson R: 0.6002 | RMSE: 2.0883


In [12]:
# train concatenation baseline

print("="*50)
print("Condition 3: Concatenation (Structure + Morphology)")
print("="*50)

print("\nTraining...")
model_concat = ConcatMLP()
model_concat, hist_concat = train_model_v2(model_concat, train_loader, val_loader, epochs=100)
r_concat, rmse_concat, _, _ = evaluate_model_v2(model_concat, test_loader)
print(f"\nConcatenation   | Pearson R: {r_concat:.4f} | RMSE: {rmse_concat:.4f}")

print("\n" + "="*50)
print("BASELINE SUMMARY")
print("="*50)
print(f"Structure-only  | Pearson R: {r_morgan:.4f} | RMSE: {rmse_morgan:.4f}")
print(f"Morphology-only | Pearson R: {r_morpho:.4f} | RMSE: {rmse_morpho:.4f}")
print(f"Concatenation   | Pearson R: {r_concat:.4f} | RMSE: {rmse_concat:.4f}")

Condition 3: Concatenation (Structure + Morphology)

Training...
  Epoch  10 | train MSE: 2.3521 | val MSE: 3.7789
  Early stopping at epoch 12

Concatenation   | Pearson R: 0.6085 | RMSE: 2.0542

BASELINE SUMMARY
Structure-only  | Pearson R: 0.3333 | RMSE: 2.4652
Morphology-only | Pearson R: 0.6002 | RMSE: 2.0883
Concatenation   | Pearson R: 0.6085 | RMSE: 2.0542


In [14]:
# save baseline results

import json

results = {
    'structure_only':  {'pearson_r': float(r_morgan), 'rmse': float(rmse_morgan)},
    'morphology_only': {'pearson_r': float(r_morpho), 'rmse': float(rmse_morpho)},
    'concatenation':   {'pearson_r': float(r_concat),  'rmse': float(rmse_concat)}
}

(BASE_DIR / "results").mkdir(parents=True, exist_ok=True)

with open(BASE_DIR / "results" / "baseline_results.json", 'w') as f:
    json.dump(results, f, indent=2)

torch.save(model_morgan.state_dict(), BASE_DIR / "results" / "model_morgan.pt")
torch.save(model_morpho.state_dict(), BASE_DIR / "results" / "model_morpho.pt")
torch.save(model_concat.state_dict(), BASE_DIR / "results" / "model_concat.pt")

print("Saved: baseline_results.json")
print("Saved: model_morgan.pt, model_morpho.pt, model_concat.pt")

Saved: baseline_results.json
Saved: model_morgan.pt, model_morpho.pt, model_concat.pt


In [20]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
from pathlib import Path

BASE_DIR = Path("..")
PROC_DIR = BASE_DIR / "data" / "processed"

master = pd.read_csv(PROC_DIR / "training_table.csv")

all_cols    = master.columns.tolist()
morgan_cols = [c for c in all_cols if c.startswith('morgan_')]
morpho_cols = [c for c in all_cols if c.startswith('Cells_')
               or c.startswith('Nuclei_')
               or c.startswith('Cytoplasm_')]
ic50_cols   = [c for c in all_cols
               if c not in morgan_cols + morpho_cols + ['drug_name', 'Metadata_JCP2022']]

drug_names   = master['drug_name'].values
drug_index   = {name: i for i, name in enumerate(drug_names)}
morgan_lookup = master.set_index('drug_name')[morgan_cols].values
morpho_lookup = master.set_index('drug_name')[morpho_cols].values

class DrugResponseDataset(Dataset):
    def __init__(self, df, drug_index, morgan_scaled, morpho_scaled):
        self.drug_idx = torch.tensor(
            [drug_index[d] for d in df['drug_name']], dtype=torch.long
        )
        self.targets = torch.tensor(df['ln_ic50'].values, dtype=torch.float32)
        self.morgan  = torch.tensor(morgan_scaled, dtype=torch.float32)
        self.morpho  = torch.tensor(morpho_scaled, dtype=torch.float32)
    def __len__(self):
        return len(self.targets)
    def __getitem__(self, i):
        idx = self.drug_idx[i]
        return self.morgan[idx], self.morpho[idx], self.targets[i]

def train_model_v2(model, train_loader, val_loader, epochs=100, lr=1e-3, patience=15):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = nn.MSELoss()
    best_val, best_state, patience_counter = float('inf'), None, 0
    for epoch in range(epochs):
        model.train()
        for morgan_b, morpho_b, target_b in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(morgan_b, morpho_b), target_b)
            loss.backward()
            optimizer.step()
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for morgan_b, morpho_b, target_b in val_loader:
                val_loss += criterion(model(morgan_b, morpho_b), target_b).item() * len(target_b)
        val_loss /= len(val_loader.dataset)
        scheduler.step(val_loss)
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    model.load_state_dict(best_state)
    return model, None

def evaluate_model_v2(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for morgan_b, morpho_b, target_b in loader:
            preds.extend(model(morgan_b, morpho_b).numpy())
            targets.extend(target_b.numpy())
    preds, targets = np.array(preds), np.array(targets)
    r, _ = pearsonr(preds, targets)
    rmse  = np.sqrt(((preds - targets)**2).mean())
    return r, rmse, preds, targets

print(f"Drug names: {len(drug_names)}")
print(f"Morgan cols: {len(morgan_cols)}")
print(f"Morpho cols: {len(morpho_cols)}")
print(f"IC50 cols: {len(ic50_cols)}")

Drug names: 175
Morgan cols: 2048
Morpho cols: 3178
IC50 cols: 969


In [21]:
# adding multi-seed training

from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

SEEDS = [42, 0, 1, 7, 21]

class MorganMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2048, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 1)
        )
    def forward(self, morgan, morpho):
        return self.net(morgan).squeeze(-1)

class MorphoMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3178, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256),  nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),  nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 1)
        )
    def forward(self, morgan, morpho):
        return self.net(morpho).squeeze(-1)

class ConcatMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5226, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256),  nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),  nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 1)
        )
    def forward(self, morgan, morpho):
        return self.net(torch.cat([morgan, morpho], dim=1)).squeeze(-1)


def run_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    shuffled_idx = np.random.permutation(len(drug_names))
    n_test  = int(len(drug_names) * 0.15)
    n_val   = int(len(drug_names) * 0.15)
    n_train = len(drug_names) - n_test - n_val

    train_drugs = set(drug_names[shuffled_idx[:n_train]])
    val_drugs   = set(drug_names[shuffled_idx[n_train:n_train+n_val]])
    test_drugs  = set(drug_names[shuffled_idx[n_train+n_val:]])

    ic50_long = master[['drug_name'] + ic50_cols].melt(
        id_vars='drug_name', var_name='cell_line', value_name='ln_ic50'
    ).dropna(subset=['ln_ic50']).reset_index(drop=True)

    train_df = ic50_long[ic50_long['drug_name'].isin(train_drugs)].reset_index(drop=True)
    val_df   = ic50_long[ic50_long['drug_name'].isin(val_drugs)].reset_index(drop=True)
    test_df  = ic50_long[ic50_long['drug_name'].isin(test_drugs)].reset_index(drop=True)

    train_idx    = [drug_index[d] for d in train_drugs]
    morpho_scaler = StandardScaler()
    morpho_sc    = morpho_scaler.fit(morpho_lookup[train_idx]).transform(morpho_lookup)
    morgan_sc    = morgan_lookup.astype(np.float32)

    train_ds = DrugResponseDataset(train_df, drug_index, morgan_sc, morpho_sc)
    val_ds   = DrugResponseDataset(val_df,   drug_index, morgan_sc, morpho_sc)
    test_ds  = DrugResponseDataset(test_df,  drug_index, morgan_sc, morpho_sc)

    train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False)

    seed_results = {}
    for name, model in [
        ('structure_only',  MorganMLP()),
        ('morphology_only', MorphoMLP()),
        ('concatenation',   ConcatMLP())
    ]:
        model, _ = train_model_v2(model, train_loader, val_loader, epochs=100, patience=15)
        r, rmse, _, _ = evaluate_model_v2(model, test_loader)
        seed_results[name] = {'r': float(r), 'rmse': float(rmse)}

    return seed_results


# Run all seeds
all_results = {name: {'r': [], 'rmse': []} 
               for name in ['structure_only', 'morphology_only', 'concatenation']}

for seed in SEEDS:
    print(f"\nSeed {seed}...")
    res = run_seed(seed)
    for name in all_results:
        all_results[name]['r'].append(res[name]['r'])
        all_results[name]['rmse'].append(res[name]['rmse'])
    print(f"  structure_only:  R={res['structure_only']['r']:.4f}")
    print(f"  morphology_only: R={res['morphology_only']['r']:.4f}")
    print(f"  concatenation:   R={res['concatenation']['r']:.4f}")

print("\n" + "="*55)
print("MULTI-SEED RESULTS (mean ± std, n=5 seeds)")
print("="*55)
for name in all_results:
    r_mean    = np.mean(all_results[name]['r'])
    r_std     = np.std(all_results[name]['r'])
    rmse_mean = np.mean(all_results[name]['rmse'])
    rmse_std  = np.std(all_results[name]['rmse'])
    print(f"  {name:20s} | R: {r_mean:.3f} ± {r_std:.3f} | RMSE: {rmse_mean:.3f} ± {rmse_std:.3f}")


Seed 42...
  structure_only:  R=0.3006
  morphology_only: R=0.5625
  concatenation:   R=0.6137

Seed 0...
  structure_only:  R=0.1348
  morphology_only: R=0.5786
  concatenation:   R=0.6022

Seed 1...
  structure_only:  R=0.2875
  morphology_only: R=0.4187
  concatenation:   R=0.4283

Seed 7...
  structure_only:  R=0.1761
  morphology_only: R=0.4742
  concatenation:   R=0.5355

Seed 21...
  structure_only:  R=0.2596
  morphology_only: R=0.5206
  concatenation:   R=0.5629

MULTI-SEED RESULTS (mean ± std, n=5 seeds)
  structure_only       | R: 0.232 ± 0.065 | RMSE: 2.479 ± 0.251
  morphology_only      | R: 0.511 ± 0.059 | RMSE: 2.106 ± 0.230
  concatenation        | R: 0.549 ± 0.066 | RMSE: 2.023 ± 0.218


In [22]:
import json

multiseed_results = {}
for name in all_results:
    multiseed_results[name] = {
        'r_mean':    round(float(np.mean(all_results[name]['r'])), 4),
        'r_std':     round(float(np.std(all_results[name]['r'])), 4),
        'rmse_mean': round(float(np.mean(all_results[name]['rmse'])), 4),
        'rmse_std':  round(float(np.std(all_results[name]['rmse'])), 4),
        'r_per_seed': [round(float(r), 4) for r in all_results[name]['r']],
        'seeds': SEEDS
    }

with open(BASE_DIR / "results" / "multiseed_results.json", 'w') as f:
    json.dump(multiseed_results, f, indent=2)

print("Saved: multiseed_results.json")

Saved: multiseed_results.json


In [24]:
# adding Ridge Regression baseline

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

print("="*55)
print("Ridge Regression Baselines (drug-level)")
print("="*55)

ridge_per_seed = {name: {'r': [], 'rmse': []}
                  for name in ['structure_only', 'morphology_only', 'concatenation']}

for seed in SEEDS:
    np.random.seed(seed)

    shuffled_idx = np.random.permutation(len(drug_names))
    n_test  = int(len(drug_names) * 0.15)
    n_val   = int(len(drug_names) * 0.15)
    n_train = len(drug_names) - n_test - n_val

    train_drugs = list(drug_names[shuffled_idx[:n_train]])
    test_drugs  = list(drug_names[shuffled_idx[n_train+n_val:]])

    train_idx     = [drug_index[d] for d in train_drugs]
    test_idx      = [drug_index[d] for d in test_drugs]

    morpho_scaler = StandardScaler()
    morpho_sc     = morpho_scaler.fit(morpho_lookup[train_idx]).transform(morpho_lookup)
    morgan_sc     = morgan_lookup.astype(np.float32)

    # IC50 matrix — mean IC50 per drug as target (drug-level prediction)
    ic50_matrix = pd.read_csv(PROC_DIR / "gdsc2_ic50_matrix.csv", index_col=0)
    y_all = ic50_matrix.mean(axis=1).values  # mean IC50 across cell lines per drug

    y_train = y_all[train_idx]
    y_test  = y_all[test_idx]

    features = {
        'structure_only':  (morgan_sc[train_idx],  morgan_sc[test_idx]),
        'morphology_only': (morpho_sc[train_idx],  morpho_sc[test_idx]),
        'concatenation':   (np.hstack([morgan_sc[train_idx], morpho_sc[train_idx]]),
                            np.hstack([morgan_sc[test_idx],  morpho_sc[test_idx]]))
    }

    for name, (X_tr, X_te) in features.items():
        ridge = Ridge(alpha=1.0)
        ridge.fit(X_tr, y_train)
        y_pred = ridge.predict(X_te)
        r, _   = pearsonr(y_pred, y_test)
        rmse   = np.sqrt(((y_pred - y_test)**2).mean())
        ridge_per_seed[name]['r'].append(float(r))
        ridge_per_seed[name]['rmse'].append(float(rmse))

print("\nRidge Regression — drug-level mean IC50 (mean ± std, n=5 seeds):")
print(f"{'Condition':22s} | {'Pearson R':>16} | {'RMSE':>14}")
print("-"*58)
for name in ridge_per_seed:
    r_mean    = np.mean(ridge_per_seed[name]['r'])
    r_std     = np.std(ridge_per_seed[name]['r'])
    rmse_mean = np.mean(ridge_per_seed[name]['rmse'])
    rmse_std  = np.std(ridge_per_seed[name]['rmse'])
    print(f"  {name:20s} | R: {r_mean:.3f} ± {r_std:.3f} | RMSE: {rmse_mean:.3f} ± {rmse_std:.3f}")

print("\nMLP — cell-line-level IC50 (mean ± std, n=5 seeds):")
print("-"*58)
for name in ['structure_only', 'morphology_only', 'concatenation']:
    r_mean    = np.mean(all_results[name]['r'])
    r_std     = np.std(all_results[name]['r'])
    rmse_mean = np.mean(all_results[name]['rmse'])
    rmse_std  = np.std(all_results[name]['rmse'])
    print(f"  {name:20s} | R: {r_mean:.3f} ± {r_std:.3f} | RMSE: {rmse_mean:.3f} ± {rmse_std:.3f}")

Ridge Regression Baselines (drug-level)

Ridge Regression — drug-level mean IC50 (mean ± std, n=5 seeds):
Condition              |        Pearson R |           RMSE
----------------------------------------------------------
  structure_only       | R: 0.012 ± 0.270 | RMSE: 2.313 ± 0.621
  morphology_only      | R: 0.010 ± 0.097 | RMSE: 2.973 ± 0.447
  concatenation        | R: 0.012 ± 0.136 | RMSE: 2.660 ± 0.393

MLP — cell-line-level IC50 (mean ± std, n=5 seeds):
----------------------------------------------------------
  structure_only       | R: 0.232 ± 0.065 | RMSE: 2.479 ± 0.251
  morphology_only      | R: 0.511 ± 0.059 | RMSE: 2.106 ± 0.230
  concatenation        | R: 0.549 ± 0.066 | RMSE: 2.023 ± 0.218


In [25]:
import json

final_results = {
    'mlp_multiseed': multiseed_results,
    'ridge_druglevel': {
        name: {
            'r_mean':    round(float(np.mean(ridge_per_seed[name]['r'])), 4),
            'r_std':     round(float(np.std(ridge_per_seed[name]['r'])), 4),
            'rmse_mean': round(float(np.mean(ridge_per_seed[name]['rmse'])), 4),
            'rmse_std':  round(float(np.std(ridge_per_seed[name]['rmse'])), 4),
        }
        for name in ridge_per_seed
    },
    'single_seed_models': {
        'cross_attention': {'pearson_r': 0.5972, 'rmse': 2.0656},
        'gated_fusion':    {'pearson_r': 0.5848, 'rmse': 2.0561}
    },
    'notes': {
        'mlp_seeds': SEEDS,
        'ridge_task': 'drug-level mean IC50',
        'mlp_task': 'cell-line-level IC50'
    }
}

with open(BASE_DIR / "results" / "final_results.json", 'w') as f:
    json.dump(final_results, f, indent=2)

print("Saved: final_results.json")
print("\nAll experiments complete.")
print("\nFinal paper results:")
print(f"  Structure-only:  R = 0.232 ± 0.065")
print(f"  Morphology-only: R = 0.511 ± 0.059")
print(f"  Concatenation:   R = 0.549 ± 0.066")
print(f"  Cross-Attention: R = 0.597 (single seed)")
print(f"  Gated Fusion:    R = 0.585 (single seed)")

Saved: final_results.json

All experiments complete.

Final paper results:
  Structure-only:  R = 0.232 ± 0.065
  Morphology-only: R = 0.511 ± 0.059
  Concatenation:   R = 0.549 ± 0.066
  Cross-Attention: R = 0.597 (single seed)
  Gated Fusion:    R = 0.585 (single seed)


In [4]:
# Statistical significance testing — paired t-test across 5 seeds
# Load from saved results

import json
import numpy as np
from scipy.stats import ttest_rel

with open("../results/final_results.json") as f:
    final = json.load(f)

r_structure  = final['mlp_multiseed']['structure_only']['r_per_seed']
r_morphology = final['mlp_multiseed']['morphology_only']['r_per_seed']
r_concat     = final['mlp_multiseed']['concatenation']['r_per_seed']

t1, p1 = ttest_rel(r_morphology, r_structure)
t2, p2 = ttest_rel(r_concat, r_structure)
t3, p3 = ttest_rel(r_concat, r_morphology)

print(f"Morphology vs Structure: t={t1:.2f}, p={p1:.3f}")
print(f"Concat vs Structure:     t={t2:.2f}, p={p2:.3f}")
print(f"Concat vs Morphology:    t={t3:.2f}, p={p3:.3f}")

Morphology vs Structure: t=5.58, p=0.005
Concat vs Structure:     t=6.00, p=0.004
Concat vs Morphology:    t=4.02, p=0.016
